# Implementing `Orthogonal Nonnegative Matrix Tri-Factorizations for Clustering` From Scratch

## Section 1: Foundations of Nonnegative Matrix Factorization (NMF)

### 1.1 The Intuition: From Data to Parts-Based Representation

Matrix factorization techniques are fundamental tools in machine learning and data analysis which are designed to decompose a complex data matrix into a product of lower-rank matrices.<br>
Methods like Principal Component Analysis or as we call PCA find orthogonal directions of maximum variance in the data. However the resulting factors often contain mixed positive and negative values which can complicate their interpretation.<br>
**Nonnegative Matrix Factorization (NMF)** offers a distinct and powerful alternative from PCA. Its core principle is the decomposition of a nonnegative data matrix into the product of two other nonnegative matrices. This constraint of nonnegativity is not just a typical mathematical constraint but it fundamentally changes the nature of the decomposition. Instead of finding abstract variance-maximizing components NMF learns an additive, "parts-based" representation of the data.<br>

A classic illustration involves decomposing a matrix of face images, where NMF can learn to identify constituent parts like eyes, noses, and mouths. The original faces can then be reconstructed by additively combining these learned parts. This inherent interpretability has made NMF a valuable technique across diverse fields, including:
- Text mining (e.g., topic modeling)
- Bioinformatics (e.g., gene expression analysis)
- Pattern recognition

### 1.2 Mathematical Formulation of Standard 2-Factor NMF
Formally the standard 2-factor NMF problem seeks to approximate a given nonnegative data matrix $X \in {R}^{p \times n}$ by the product of two lower-rank nonnegative matrices:$F \in {R}^{p \times k}$ and $G \in \mathbb{R}^{n \times k}.$ <br>
Here:
- $p$ is the number of features (e.g., words in a vocabulary)
- $n$ is the number of samples (e.g., documents)
- $k$ is the number of latent components or topics which is typically $k \ll \min(p, n).$

The approximation is expressed as $X \approx F G^T.$


The goal is to find the factors $F$ and $G$ that minimize the reconstruction error between the original matrix $X$ and its approximation $FG^T.$ The most common objective function for this purpose is the squared Frobenius norm of the difference:  $\min_{F \geq 0, G \geq 0} \|X - F G^T\|_F^2.$<br>
The Frobenius norm $\|A\|_F$ is the square root of the sum of the squares of all elements of $A$: $\|A\|_F = \sqrt{ \sum_{i,j} A_{ij}^2 }$ <br>
Minimizing this objective function is equivalent to minimizing the sum of squared errors between each element of $X$ and its reconstructed counterpart in $FG^T.$ The matrices $F$ and $G$ are often referred to as the basis (or "features") matrix and the coefficients (or "encodings") matrix respectively.


#### The Challenge of Non-Uniqueness

A significant challenge with the standard NMF formulation is the non-uniqueness of its solution. For any given solution pair $(F,G)$ there exists a large set of alternative solutions that produce the exact same reconstruction error.<br>
Specifically for any invertible matrix $A$ such that both $F A$ and $(G(A^{-1})^T$ remains non-negative then the pair $F A, (G(A^{-1})^T$ is also a valid solution<br> $X \approx (FA)(G(A^{-1})^T)^T = FAGA^{-1} = F G^T $<br>

This non-uniqueness is not just a mathematical inconvenience but also it strikes at the core of NMF's practical value. The primary appeal of NMF lies in its promise of interpretable factors. For example, in text analysis a column in the factor matrix $F$ might be interpreted as a specific **topic** due to its high weights on related words. This interpretation is only meaningful however only if if the factors are stable and unique.<br>
If an equally valid solution exists where that **topic** is smeared across multiple columns or mixed with other topics the initial interpretation becomes arbitrary and unreliable. The ambiguity of the solution undermines the very interpretability that motivates the use of NMF in the first place. This fundamental problem makes necessary the introduction of additional constraints to guide the factorization towards a more meaningful and unique solution.

## Section 2: Introducing Orthogonality- A Bridge to Clustering

### 2.1 Uni-Orthogonal NMF: Formulation and Motivation

A powerful method for resolving the non-uniqueness issue in NMF is to impose orthogonality constraints on one of the factor matrices. This constraint drastically reduces the space of possible solutions. As shown in the paper with an orthogonality condition such as $G^T G = I$ (where $I$ is the identity matrix) the only remaining ambiguity in the solution is a trivial permutation of the columns which does not affect the interpretation of the individual factors. The two primary advantages of this approach are the uniqueness of the solution and the emergence of rigorous clustering interpretations. <br>
Here we will focus on G-orthogonal NMF, where the constraint is applied to the coefficient matrix $G.$ The optimization problem is formulated as follows:<br>
$\min_{F \geq 0, G \geq 0} \|X - F G^T\|_F^2 \quad \text{s.t. } G^T G = I$<br>

Heere the columns of $G$ are constrained to be orthogonal to each other and this formulation is referred to as uni-orthogonal NMF.



### 2.2 The Equivalence of Orthogonal NMF and K-Means Clustering

One of the greatest consequences of introducing an orthogonality constraint is the establishment of a direct equivalence between NMF and K-means clustering. This connection provides a solid theoretical foundation for using NMF as a clustering algorithm. <br>
In the paper **Theorem 1**  states that G-orthogonal NMF is equivalent to K-means clustering.<br>
Below we have proof which begins by expanding the objective function:<br>

$J = ||X - F G^T||_F^2 :$<br>

$J = \text{Tr}((X -FG^T)^T(X - FG^T)) = \text{Tr}(X^T X - 2X^T F G^T + G F^T F G^T)$

Using the cyclic property of the trace of the matrix, above equation can be written as follows:

$ J= \text{Tr}(X^T X) - 2 \text{Tr}(G^T X^T F) + \text{Tr}(F^T F G^T G)$

Since $G^T G = I$ then the expression becomes as:

$J = \text{Tr}(X^T X) - 2 \text{Tr}(G^T X^T F) + \text{Tr}(F^T F)$

Now to find the optimal $F$ for a fixed $G$ the gradient of $J$ with respect to $F$ is set to zero as follows:

$\frac{\partial J}{\partial F} = -2 X G + 2 F = 0 \quad \Rightarrow \quad F = X G $

Substituting this optimal $F$ back into the objective function finally gives:

$ J = \text{Tr}(X^T X) -2\text{Tr}(G^T X^T (X G)) + \text{Tr}((X G)^T(X G))$

$ J = \text{Tr}(X^T X) -2\text{Tr}(G^T X^T X G) + \text{Tr}(G^T X^T X G)$

$J = \text{Tr}(X^T X) - \text{Tr}(G^T X^T X G)$

Since $\text{Tr}(X^T X)$ is constant, minimizing $J$ is equivalent to:

$\max_{G \geq 0, G^T G = I} \text{Tr}(G^T X^T X G)$

which results into an optimization problem which is a well-known form of spectral relaxation for K-means clustering. Here $G$ acts as a "soft" cluster indicator matrix, where the element $G_{ji}$ represents the degree of membership of document $j$ in cluster $i$.


The orthogonality constraint $G^T G=I$ is not merely a mathematical convenience that simplifies the objective function. It also imposes a powerful structural prior on the matrix $G.$<br>
For a non-negative matrix this constraint forces its columns to be mutually orthogonal and have a unit L2-norm. In an ideal **hard clustering** scenario this would mean that each row of $G$ could have only one non-zero entry, making $G$ a discrete cluster indicator matrix where each sample belongs to exactly one cluster.<br> In practice the algorithm produces a **soft** clustering but the constraint is the fundamental mechanism that transforms $G$ from a continuous factor matrix into a matrix that explicitly represents cluster assignments. This insight reveals how an algebraic constraint on a matrix factorization problem directly gives rise to emergent clustering behavior.

## Section 3: Practical Implementation of Uni-Orthogonal NMF



### 3.1 Data Acquisition and Preprocessing

To implement the algorithm in a practical from scratch we use a subset of the 20 Newsgroups dataset which is a classic corpus for text classification and clustering experiments.<br>
To create a manageable yet interesting problem four distinct categories are selected:
`rec.sport.baseball`, `sci.crypt`, `sci.med`, and `talk.politics.misc`. This mix of topics provides us a suitable challenge for evaluating the clustering algorithm's ability to separate documents. <br>

Our implementation follows a standard text preprocessing pipeline which is essential for transforming raw text into a numerical format suitable for matrix factorization.

1. **Loading Data:** The dataset is loaded using the `sklearn.datasets.fetch_20newsgroups` which is taken from [Kaggle Dataset](https://www.kaggle.com/code/giangpt/20-news-group-dataset-starter) which provides easy access to the text data and their corresponding ground-truth labels.

2. **Cleaning Text:** Raw text from newsgroups often contains metadata headers, footers, and quoted replies that can act as noise and potentially lead to trivial clustering solutions. These elements are removed to ensure the clustering is based on the core content of the documents.

3. **Vectorization:** The cleaned text documents are converted into a numerical term-document matrix which is denoted as $X$. This is achieved using `TfidfVectorizer` from [scikit-learn](https://scikit-learn.org/stable/auto_examples/text/plot_document_clustering.html). [TF-IDF (Term Frequency-Inverse Document Frequency)](https://spotintelligence.com/2023/01/17/text-clustering-algorithms/) is a weighting scheme that reflects the importance of a word to a document in a collection or corpus. It increases the weight for words that appear frequently in a document but are rare across the entire corpus which effectively filters common words and highlighting distinctive topic-specific terms.<br>
The resulting matrix $X$ is sparse, with dimensions $p×n$ where $p$ is the vocabulary size and $n$ is the number of documents.



In [1]:
import numpy as np
import re
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# To load the 20 Newsgroups dataset for 4 selected categories which cleans the text, and converts it to a TF-IDF matrix.
def load_and_preprocess_data():
    
    categories = ['rec.sport.baseball', 'sci.crypt', 'sci.med', 'talk.politics.misc'] # Selecting categories to create a manageable and interesting clustering problem
    print("Loading 20 Newsgroups dataset for categories:", categories)# loading the dataset
    dataset = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42,
                                 remove=('headers', 'footers', 'quotes')) # removing headers, footers, and quotes to clean the text data
    documents = dataset.data # Extracting the text documents from the dataset
    true_labels = dataset.target # Extracting the ground trusth labels true from dataset.target for evaluation
    """
    Here we are initializing the TF-IDF vectorizer with parameters to filter out common words and very rare words
    1. max_df=0.5: This means that words that appear in more than 50% of the documents will be ignored.
    2. min_df=5: This means that words that appear in fewer than 5 documents will be ignored.
    3. stop_words='english': This means that common English stop words (like 'the', 'is', etc.) will be ignored or removed
    """
    vectorizer = TfidfVectorizer(max_df=0.5, min_df=5, stop_words='english')
    
    print("Vectorizing text data with TF-IDF!!!")
    X = vectorizer.fit_transform(documents)# Converting the text documents into a TF-IDF matrix
    
    """
    The NMTF Paper's formulation is X ≈ FG^T, where X is (features x samples) or (p x n).
    scikit-learn's output is (samples x features) or (n x p). We need to transpose it to match the NMTF formulation.
    """
    X = X.T  # Transposing the matrix to match the NMTF formulation
    X[X < 0] = 0  # Ensuring all values are non-negative which is required by NMTF
    print(f"Data matrix X created with shape (words x documents): {X.shape}")
    return X, true_labels, dataset.target_names

X, true_labels, target_names = load_and_preprocess_data()

Loading 20 Newsgroups dataset for categories: ['rec.sport.baseball', 'sci.crypt', 'sci.med', 'talk.politics.misc']
Vectorizing text data with TF-IDF:
Data matrix X created with shape (words x documents): (8292, 3750)


### 3.2 From-Scratch Algorithm Implementation

The core of the uni-orthogonal NMF algorithm is its set of multiplicative update rules. These rules are derived from the objective function and its constraints and they guarantee a monotonic decrease in the reconstruction error which ensures convergence to a local minimum. The implementation below is a faithful reproduction of the update rules for G-orthogonal NMF as presented in the paper.<br>

For $F$:

$F_{ik} \leftarrow F_{ik} \frac{(XG)_{ik}}{(F G^T G)_{ik}}$

Since $G^T G = I$ which simplifies to:
$[
F_{ik} \leftarrow F_{ik} \frac{(XG)_{ik}}{F_{ik}} = (XG)_{ik}
]$

However the paper uses the **standard NMF update rule** for $F$ which is more stable in practice:<br>

$F_{ik} \leftarrow F_{ik} \frac{(XG)_{ik}}{(F G^T G)_{ik}}$

For $G$:

$G_{jk} \leftarrow G_{jk} \frac{(X^T F)_{jk}}{(G G^T X^T F)_{jk}}$


Our implementation contrasts with the solvers available in standard libraries like scikit-learn which often uses more complex methods like Coordinate Descent or [Projected Gradient Descent](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html). By implementing these specific multiplicative rules our analysis remains true to the paper's methodology.

In [ ]:
"""
Implementing Uni-Orthogonal NMF with the constraint G^T * G = I.
Following are the arguments for the function:
- X: The input data matrix (p x n) where p is the number of features and n is the number of samples.
- k(ubt): The number of components(cluster/topics) to factorize X into.
- max_iter: The maximum number of iterations for the algorithm to run.
- tol: The tolerance for convergence. If the change in the Frobenius norm of the difference between the current and previous iteration is less than tol then the algorithm will stop.
"""
def uni_orthogonal_nmf(X, k, max_iter=100, tol=1e-4):
    p, n = X.shape # Getting the shape of the input matrix X
    
    # Initializing F and G with non-negative random values and ising a small value to avoid division by zero
    F = np.random.rand(p, k) + 1e-9
    G = np.random.rand(n, k) + 1e-9
    
    # converting sparse matrix to dense for calculations if needed
    if hasattr(X, "toarray"): # Check if X is a sparse matrix
        X_dense = X.toarray()# Converting sparse matrix to dense
    else:
        X_dense = X# Using the dense matrix directly if it is already dense
    
    prev_error = np.inf  # Initializing previous error to infinity for convergence check
    
    print(f"Starting Uni-Orthogonal NMF for k={k} clusters with max_iter={max_iter} and tol={tol}")
    for i in range(max_iter):
        # Update G using the multiplicative update rule
        numerator_F = X_dense @ G
        denominator_F = F @ (G.T @ G)
        # Since G^T @ G = I this simplifies to F like in paper
        F *= numerator_F / (denominator_F + 1e-9)  # adding epsilon for stability from the paper due to standard update which is more stable
        
        """
        Applying update rule for G with orthogonality constraint which is the special update rule from the paper
        Numerator: (X.T @ F)
        Denominator: (G @ G.T @ X.T @ F)
        """
        numerator_G = X_dense.T @ F
        denominator_G = G @ (G.T @ (X_dense.T @ F))
        G *= numerator_G / (denominator_G + 1e-9)
        
        """
        Now we normalize columns of G to enforce orthogonality.
        Although the update rule theoretically maintains it, numerical stability can be a big issue.
        This step is a form of projected gradient descent.
        For simplicity in this implementation from the paper we will rely on the update rule.
        """
        
        error = np.linalg.norm(X_dense - F @ G.T, 'fro')# Calculating the Frobenius norm of the difference between X and FG^T for convergence 
        if abs(prev_error - error) < tol:# Checking if the change in error is less than the tolerance
            print(f"Convergence reached at iteration {i+1} with error {error:.4f}")
            break
        prev_error = error # Updating the previous error for the next iteration
        
        if (i + 1) % 10 == 0:# Printing the error every 10 iterations
            print(f"Iteration {i+1}/{max_iter}, Reconstruction Error: {error:.4f}")
    
    return F, G  # Returning the factor matrices F and G

### 3.3 Training and Evaluation of Document Clusters

After implementing the algorithm our next step is to train it on the preprocessed data and evaluate the quality of the resulting document clusters. The training process involves iteratively applying the update rules until the reconstruction error converges. The matrix $G$ of shape (documents × clusters) contains the cluster assignments. For a given document (a row in $G$), the cluster with the highest coefficient is its assigned cluster.

To quantitatively assess performance the predicted cluster labels are compared against the ground-truth newsgroup labels from the dataset using standard clustering metrics. By calculating both Purity and ARI from the reference paper our evaluation provides a more complete and critical picture of the algorithm's performance.

- **Purity:** This metric measures the extent to which each cluster contains documents from a single class. To calculate [purity](https://permetrics.readthedocs.io/en/latest/pages/clustering/PuS.html) each cluster is assigned the label of the most frequent true class within it. The purity is then the fraction of correctly assigned documents. A simple way to compute this involves using scikit-learn's [contingency_matrix.](https://towardsdatascience.com/evaluation-metrics-for-clustering-models-5dde821dd6cd/)
- **Adjusted Rand Index (ARI):** Most of the time purity has a significant flaw i.e. it tends to increase with the number of clusters and can be trivially maximized by assigning each data point to its own cluster. The [Adjusted Rand Index (ARI)](https://www.numberanalytics.com/blog/master-adjusted-rand-index-guide) is a more robust metric that corrects for this by accounting for chance groupings. An [ARI score](https://www.kaggle.com/code/metric/adjusted-rand-score) of 1.0 indicates a perfect match while a score close to 0.0 suggests the clustering is no better than random. This ["adjustment for chance"](https://www.geeksforgeeks.org/machine-learning/clustering-performance-evaluation-in-scikit-learn/) makes ARI a superior choice for rigorous performance comparisons. 



In [5]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics.cluster import contingency_matrix
from sklearn.cluster import KMeans

In [ ]:
# Calculating and returning purity and ARI scores
def evaluate_clustering(true_labels, pred_labels):
    
    # Calcuationg PURITY
    cont_matrix = contingency_matrix(true_labels, pred_labels)# Contingency matrix for true and predicted labels
    purity = np.sum(np.amax(cont_matrix, axis=0)) / np.sum(cont_matrix)  # Purity is the sum of the maximum values in each column divided by the total
    
    # Adjusted Rand Index
    ari = adjusted_rand_score(true_labels, pred_labels)
    
    return purity, ari